# Divye FE — 20-Seed Full-Data Average

Same Divye FE features as `s6e2-divye-fe.ipynb`, but trained on **full data** with 20 seeds averaged.

Pattern from `s6e2-cpu-seedavg-fulldata.ipynb`: seed averaging reduces variance in tree structure  
without requiring a validation split. Previously gave LB 0.95350 on the default CatBoost.

For full-data TE (no OOF): computed using the complete training set — no leakage risk since  
there's no validation set, and test predictions are clean.

Fixed iterations: based on divye-fe CV best iterations (~1100-1300 range).  
Checkpointed — resumable if interrupted.

Baseline: catboost_divye_fe OOF = 0.95566  LB (expected) ~0.9537

In [1]:
import subprocess
import time
import numpy as np
import pandas as pd
from itertools import combinations
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier

DATA_DIR = Path('data')
train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
ss    = pd.read_csv(DATA_DIR / 'sample_submission.csv')

def prep(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
    if 'heart_disease' in df.columns:
        df['heart_disease'] = df['heart_disease'].map({'Absence': 0, 'Presence': 1})
    return df

train = prep(train)
test  = prep(test)

CAT_FEATURES = ['sex', 'chest_pain_type', 'fbs_over_120', 'ekg_results',
                'exercise_angina', 'slope_of_st', 'number_of_vessels_fluro', 'thallium']
NUM_FEATURES = ['age', 'bp', 'cholesterol', 'max_hr', 'st_depression']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES

X      = train[ALL_FEATURES]
y      = train['heart_disease'].values
X_test = test[ALL_FEATURES]

print(f'Train: {X.shape}  Test: {X_test.shape}')

Train: (630000, 13)  Test: (270000, 13)


In [2]:
def compute_freq_features(train_df, test_df, cols):
    combined = pd.concat([train_df[cols], test_df[cols]])
    tr_freq, te_freq = {}, {}
    for col in cols:
        freq_map = combined[col].value_counts(normalize=True)
        tr_freq[f'{col}_freq'] = train_df[col].map(freq_map).fillna(0).values
        te_freq[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0).values
    return tr_freq, te_freq


def compute_fulldata_te(train_df, test_df, cols, y):
    """Full-data TE (no OOF needed — training on full data, only test predictions matter)."""
    global_mean = y.mean()
    tr_te, te_te = {}, {}
    for col in cols:
        te_map = pd.DataFrame({'v': train_df[col].values, 't': y}).groupby('v')['t'].mean()
        tr_te[f'{col}_te'] = train_df[col].map(te_map).fillna(global_mean).values
        te_te[f'{col}_te'] = test_df[col].map(te_map).fillna(global_mean).values
    return tr_te, te_te


def make_interaction_features(te_dict, top_cols):
    return {f'{c1}_x_{c2}': te_dict[c1] * te_dict[c2]
            for c1, c2 in combinations(top_cols, 2)}


def build_augmented(base_df, *dicts):
    df = base_df.copy().reset_index(drop=True)
    for d in dicts:
        for name, vals in d.items():
            df[name] = vals
    return df


print('Computing freq encoding...')
tr_freq, te_freq = compute_freq_features(X, X_test, ALL_FEATURES)

print('Computing full-data TE...')
tr_te, te_te = compute_fulldata_te(X, X_test, ALL_FEATURES, y)

# Top-8 by correlation — same selection as base notebook
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from sklearn.metrics import roc_auc_score

# Use full-data TE correlations to select top-8 (consistent with base notebook)
corrs = {k: abs(np.corrcoef(v, y)[0, 1]) for k, v in tr_te.items()}
top8  = sorted(corrs, key=corrs.get, reverse=True)[:8]
print(f'Top-8: {top8}')

tr_inter = make_interaction_features(tr_te, top8)
te_inter = make_interaction_features(te_te, top8)

X_aug      = build_augmented(X,      tr_freq, tr_te, tr_inter)
X_test_aug = build_augmented(X_test, te_freq, te_te, te_inter)

print(f'\nAugmented train: {X_aug.shape}  test: {X_test_aug.shape}')

Computing freq encoding...
Computing full-data TE...
Top-8: ['thallium_te', 'chest_pain_type_te', 'max_hr_te', 'number_of_vessels_fluro_te', 'st_depression_te', 'exercise_angina_te', 'slope_of_st_te', 'sex_te']

Augmented train: (630000, 67)  test: (270000, 67)


In [ ]:
N_SEEDS    = 20
ITERATIONS = 1300  # based on divye-fe CV best iterations
CKPT_DIR   = Path('submissions/divye_fe_seedavg_ckpts')
CKPT_DIR.mkdir(exist_ok=True)

BASE_PARAMS = dict(
    iterations    = ITERATIONS,
    learning_rate = 0.05,
    depth         = 6,
    task_type     = 'CPU',
    thread_count  = -1,
    cat_features  = CAT_FEATURES,
    verbose       = 0,
)

print(f'Seeds: {N_SEEDS}  Iterations: {ITERATIONS}  Checkpoints: {CKPT_DIR}')

fit_times = []
for seed in range(N_SEEDS):
    ckpt = CKPT_DIR / f'seed_{seed:02d}.npy'
    if ckpt.exists():
        print(f'seed={seed:2d}  [loaded from checkpoint]')
        continue

    t0 = time.time()
    m  = CatBoostClassifier(**BASE_PARAMS, random_seed=seed)
    m.fit(X_aug, y)
    preds = m.predict_proba(X_test_aug)[:, 1]
    np.save(ckpt, preds)

    elapsed = time.time() - t0
    fit_times.append(elapsed)
    completed = sorted(CKPT_DIR.glob('seed_*.npy'))
    remaining = N_SEEDS - len(completed)
    eta_min   = remaining * np.mean(fit_times) / 60 if fit_times else '?'
    print(f'seed={seed:2d}  {elapsed/60:.1f}min  ({len(completed)}/{N_SEEDS} done  ETA ~{eta_min:.0f}min)')

print('\nAll seeds complete.')

Seeds: 20  Iterations: 1300  Checkpoints: submissions/divye_fe_seedavg_ckpts
seed= 0  1.4min  (1/20 done  ETA ~27min)


In [1]:
ckpts      = sorted(CKPT_DIR.glob('seed_*.npy'))
all_preds  = np.stack([np.load(f) for f in ckpts])
test_preds = all_preds.mean(axis=0)

print(f'Averaged {len(ckpts)} seeds')
print(f'Prediction mean={test_preds.mean():.4f}  std={test_preds.std():.4f}')

label = f'divye_fe_seedavg_{len(ckpts)}seeds'
fname = f'submissions/{label}.csv'
sub = ss.copy()
sub['Heart Disease'] = test_preds
sub.to_csv(fname, index=False)

desc = f'{label} | cv_auc=NA'
print(f'Saved: {fname}')
print(f'desc:  {desc}')

NameError: name 'CKPT_DIR' is not defined

In [2]:
import subprocess
r = subprocess.run(
    ['kaggle', 'competitions', 'submit', '-c', 'playground-series-s6e2',
     '-f', fname, '-m', desc],
    capture_output=True, text=True
)
status = 'submitted' if 'successfully' in r.stdout.lower() else r.stdout.strip()[:120]
print(f'{label}: {status}')

NameError: name 'fname' is not defined